# Embedding basics

This notebook walks through generating embeddings from text, inspecting their dimensions, and visualising the cosine similarity between sentences. A good mental model: sentences that mean similar things end up close together in the vector space.

Run from the project root so the `src/` imports work.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.embeddings.generator import EmbeddingGenerator
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

gen = EmbeddingGenerator('fast')
print(f'Model: {gen.model_name}  |  Dimensions: {gen.dim}')

In [ ]:
sentences = [
    'The dog chased the ball across the yard.',
    'A puppy ran after a toy in the garden.',
    'Vector databases store embeddings for fast retrieval.',
    'Similarity search over dense vectors is efficient.',
    'Machine learning models can classify images.',
    'Neural networks learn patterns from data.',
]

embeddings = gen.embed_batch(sentences)
print(f'Embeddings shape: {embeddings.shape}')  # (6, 384)

In [ ]:
# Cosine similarity matrix (embeddings are already normalised, so it's just a dot product)
sim_matrix = embeddings @ embeddings.T

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    sim_matrix,
    annot=True, fmt='.2f',
    xticklabels=[s[:30] + '…' for s in sentences],
    yticklabels=[s[:30] + '…' for s in sentences],
    cmap='RdYlGn', vmin=0, vmax=1, ax=ax
)
ax.set_title('Cosine similarity between sentences')
plt.tight_layout()
plt.show()

In [ ]:
# Find the most similar pair (excluding self-similarity)
np.fill_diagonal(sim_matrix, 0)
i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f'Most similar pair ({sim_matrix[i,j]:.3f}):')
print(f'  A: {sentences[i]}')
print(f'  B: {sentences[j]}')

In [ ]:
# The cache saves re-embedding the same text on subsequent runs
from src.embeddings.cache import EmbeddingCache

cache = EmbeddingCache('.joblib_cache')
vec = cache.cached_embed(gen, 'this sentence is cached after the first call')
vec2 = cache.cached_embed(gen, 'this sentence is cached after the first call')
print('Cache hit identical:', np.allclose(vec, vec2))